# LANDFIRE Forest Canopy Cover — Comprehensive Analysis
## California Wildfire Prediction Model — Phase 1.9

**Prepared for**: Tam Air Club, UCSF, UCI, CAL FIRE Collaboration  
**Dataset**: LANDFIRE 2024 Forest Canopy Cover (CC)  
**Coverage**: CONUS raster clipped to California, reprojected to EPSG:3310  
**Resolution**: 30m native, analyzed at ~810m (27x downsample)  
**Date**: March 2026

---

### Why Canopy Cover Matters for Wildfire

Forest canopy cover is a critical driver of fire behavior through four mechanisms:

1. **Crown fire initiation** — dense canopies provide ladder fuels that allow surface fires to reach tree crowns
2. **Spotting probability** — closed canopies generate more firebrands that can ignite spot fires downwind
3. **Wind reduction** — dense canopies reduce near-surface wind speed, while canopy gaps create wind tunnels
4. **Fuel moisture conditioning** — canopy shade keeps surface fuels moister, while open areas dry faster

**Temporal caveat**: LANDFIRE CC 2024 is a snapshot incorporating 2015-2024 disturbances. Historical fires (pre-2015) occurred under different canopy conditions. This must be considered when interpreting fire overlay statistics.

This is the first CONUS-wide raster in this project, requiring clipping to California and CRS reprojection (EPSG:5070 → EPSG:3310).

### Table of Contents

| Part | Title | Key Outputs |
|------|-------|-------------|
| 1 | Introduction & Context | Dataset overview, learning objectives |
| 2 | Data Loading & Quality | CONUS metadata, CA clip, class distribution |
| 3 | Statewide Distribution | Histogram, statewide map, elevation relationship |
| 4 | Regional Patterns | Regional comparison, N-S gradient, E-W profiles |
| 5 | CC × Fire Perimeters | Burned vs unburned, by fire size, temporal |
| 6 | CC × Burn Severity | MTBS severity vs CC, by season, by decade |
| 7 | CC × Fire Size/Frequency | Frequency by CC class, size distribution |
| 8 | Spatial Visualization | Regional detail map, CC + fire overlay, Folium |
| 9 | ML Features & Summary | 800m aggregation demo, feature table, next steps |

---
## Part 1: Introduction & Context

### Learning Objectives

After completing this notebook, you will understand:

- The LANDFIRE Canopy Cover product: what it measures, its resolution, and classification scheme
- California's canopy cover distribution: where forests are dense, open, or absent
- How canopy cover relates to elevation, latitude, and geography
- The relationship between canopy cover and historical fire patterns
- How canopy cover interacts with burn severity
- How to engineer CC-based features for the ML prediction model

### About LANDFIRE CC

The LANDFIRE Forest Canopy Cover (CC) product describes the percent cover of the tree canopy in a stand. It is derived from satellite imagery (Landsat) and calibrated against field data. Values represent class midpoints:

| Value | Class | Broad Category |
|-------|-------|----------------|
| 0 | Non-Forested | Non-Forested |
| 15 | 10-20% | Open |
| 25 | 20-30% | Open |
| 35 | 30-40% | Moderate |
| 45 | 40-50% | Moderate |
| 55 | 50-60% | Moderate |
| 65 | 60-70% | Dense |
| 75 | 70-80% | Dense |
| 85 | 80-90% | Very Dense |
| 95 | 90-100% | Very Dense |

---
## Part 2: Data Loading & Quality

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────

# Standard
import sys
import warnings
warnings.filterwarnings('ignore')

# Data
import numpy as np
import pandas as pd
import geopandas as gpd

# Raster
import rasterio
from rasterio.transform import from_bounds

# Visualization
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
import contextily as ctx

# Project utilities
sys.path.insert(0, '..')
from src.utils.constants import (
    LANDFIRE_CC_PATH, LANDFIRE_CC_CSV_PATH, LANDFIRE_CC_NODATA,
    LANDFIRE_CC_RESOLUTION_M, LANDFIRE_CC_VIZ_DOWNSAMPLE,
    CRS_CONUS_ALBERS, CRS_ALBERS, CRS_WGS84, CA_BOUNDARY_PATH,
    FIRE_PERIMETERS_PATH, MTBS_FOD_PATH, HIGH_QUALITY_CUTOFF,
    FIGURES_DIR, FIGURE_DPI, OUTPUT_DIR,
)
from src.utils.landfire_domain_mappings import (
    CC_VALUE_MAP, CC_MIDPOINT_MAP, CC_ORDERED_VALUES, CC_RGB_MAP,
    CC_BROAD_CATEGORIES, CC_BROAD_CATEGORY_ORDER, CC_BROAD_CATEGORY_COLORS,
    get_cc_label, get_cc_color, get_cc_broad_category,
    build_cc_colormap, build_cc_norm, classify_canopy_cover,
)
from src.utils.landfire_data_loader import (
    load_landfire_cc_metadata, load_cc_california,
    load_cc_downsampled, extract_cc_at_points, extract_cc_for_polygons,
    get_cc_class_distribution, get_cc_data_quality_summary,
)
from src.utils.plotting import (
    setup_plotting_style, save_figure,
    get_canopy_cover_colors, get_canopy_broad_colors,
)

# Set up style
setup_plotting_style()

# Output directories
LANDFIRE_FIG_DIR = FIGURES_DIR / 'landfire_cc'
LANDFIRE_FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Imports complete.')

In [ ]:
# ── Load CONUS metadata ──────────────────────────────────────────────────
meta = load_landfire_cc_metadata()

print('LANDFIRE CC CONUS Raster Metadata')
print('=' * 50)
for k, v in meta.items():
    print(f'  {k:20s}: {v}')

In [ ]:
# ── Clip to California & reproject ────────────────────────────────────────
# This is the core new capability: CONUS → CA clip + EPSG:5070 → EPSG:3310
# Memory: ~2-3 GB peak during reprojection

cc_array, cc_transform, cc_meta = load_cc_california()

print(f'\nFinal California CC array:')
print(f'  Shape: {cc_array.shape}')
print(f'  dtype: {cc_array.dtype}')
print(f'  Resolution: {cc_meta["resolution_m"]:.2f}m')
print(f'  Memory: {cc_array.nbytes / 1e9:.2f} GB')

In [ ]:
# ── Data quality summary ──────────────────────────────────────────────────
quality = get_cc_data_quality_summary(cc_array)
print('\nData Quality Summary')
print('=' * 50)
for k, v in quality.items():
    print(f'  {k:25s}: {v}')

# Class distribution
cc_dist = get_cc_class_distribution(cc_array, resolution_m=cc_meta['resolution_m'])
print('\nCC Class Distribution:')
print(cc_dist.to_string(index=False))

In [ ]:
# ── Figure 01: Data Quality & Class Distribution ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: CC class distribution bar chart
ax = axes[0]
cc_colors = get_canopy_cover_colors()
colors = [cc_colors.get(v, '#888') for v in cc_dist['value']]
bars = ax.bar(range(len(cc_dist)), cc_dist['pct_of_valid'], color=colors, edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(cc_dist)))
ax.set_xticklabels([get_cc_label(v) for v in cc_dist['value']], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('% of Valid Pixels')
ax.set_title('CC Class Distribution (California)')
# Add percentage labels
for bar, pct in zip(bars, cc_dist['pct_of_valid']):
    if pct > 1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{pct:.1f}%', ha='center', va='bottom', fontsize=8)

# Right: Forested vs Non-Forested pie + broad categories
ax2 = axes[1]
broad_counts = cc_dist.groupby('broad_category')['pixel_count'].sum()
broad_counts = broad_counts.reindex(CC_BROAD_CATEGORY_ORDER)
broad_pcts = 100.0 * broad_counts / broad_counts.sum()
broad_colors_map = get_canopy_broad_colors()
bc = [broad_colors_map[cat] for cat in CC_BROAD_CATEGORY_ORDER]

wedges, texts, autotexts = ax2.pie(
    broad_pcts, labels=CC_BROAD_CATEGORY_ORDER, colors=bc,
    autopct='%1.1f%%', startangle=90, pctdistance=0.75,
    textprops={'fontsize': 9}
)
for t in autotexts:
    t.set_fontsize(8)
ax2.set_title('Broad Canopy Cover Categories')

fig.suptitle('Figure 1: LANDFIRE CC Data Quality & Class Distribution', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
save_figure(fig, 'landfire_cc/01_data_quality_class_distribution')
plt.show()

---
## Part 3: Statewide Distribution

California's canopy cover varies dramatically from the dense coastal redwoods and Sierra Nevada conifer forests to the vast non-forested areas of the Central Valley and Mojave Desert.

In [ ]:
# ── Downsample for visualization ──────────────────────────────────────────
cc_ds, cc_ds_transform, cc_ds_meta = load_cc_downsampled(cc_array, cc_transform)

print('Downsampled CC array:')
for k, v in cc_ds_meta.items():
    print(f'  {k}: {v}')

In [ ]:
# ── Figure 02: Canopy Cover Distribution ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Detailed histogram of CC values
ax = axes[0]
valid_vals = cc_ds[~np.isnan(cc_ds)]
# Map to midpoints for histogram
cc_colors_map = get_canopy_cover_colors()
for v in CC_ORDERED_VALUES:
    count = np.sum(cc_dist.loc[cc_dist['value'] == v, 'pixel_count'].values)
    ax.bar(v, count, width=8, color=cc_colors_map.get(v, '#888'), edgecolor='white')
ax.set_xlabel('Canopy Cover Class (Midpoint %)')
ax.set_ylabel('Pixel Count')
ax.set_title('CC Value Distribution')
ax.set_xticks(CC_ORDERED_VALUES)
ax.set_xticklabels([f'{v}%' for v in CC_ORDERED_VALUES], fontsize=9)

# Right: Cumulative distribution
ax2 = axes[1]
cum_pct = cc_dist['pct_of_valid'].cumsum()
ax2.plot(CC_ORDERED_VALUES, cum_pct.values, 'o-', color='#2E86C1', linewidth=2, markersize=8)
ax2.fill_between(CC_ORDERED_VALUES, cum_pct.values, alpha=0.2, color='#2E86C1')
ax2.set_xlabel('Canopy Cover Class (Midpoint %)')
ax2.set_ylabel('Cumulative % of Valid Pixels')
ax2.set_title('Cumulative Distribution')
ax2.set_xticks(CC_ORDERED_VALUES)
ax2.set_xticklabels([f'{v}%' for v in CC_ORDERED_VALUES], fontsize=9)
ax2.axhline(50, color='gray', linestyle='--', alpha=0.5)
ax2.set_ylim(0, 105)

fig.suptitle('Figure 2: Canopy Cover Distribution', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
save_figure(fig, 'landfire_cc/02_canopy_cover_distribution')
plt.show()

In [ ]:
# ── Figure 03: Statewide Canopy Cover Map ────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(10, 14))

# Build colormap and norm
cmap = build_cc_colormap()
norm = build_cc_norm()

# Compute extent from transform
h_ds, w_ds = cc_ds.shape
left = cc_ds_transform[2]
top = cc_ds_transform[5]
right = left + w_ds * abs(cc_ds_transform[0])
bottom = top + h_ds * cc_ds_transform[4]  # negative
extent = [left, right, bottom, top]

# Plot CC
im = ax.imshow(cc_ds, cmap=cmap, norm=norm, extent=extent, origin='upper', interpolation='nearest')

# Add basemap
try:
    ctx.add_basemap(ax, crs=CRS_ALBERS, source=ctx.providers.CartoDB.PositronNoLabels, alpha=0.3)
except Exception:
    pass

# Colorbar with class labels
cbar = plt.colorbar(im, ax=ax, shrink=0.6, pad=0.02)
cbar.set_ticks(CC_ORDERED_VALUES)
cbar.set_ticklabels([get_cc_label(v) for v in CC_ORDERED_VALUES])
cbar.set_label('Canopy Cover Class')

ax.set_title('Figure 3: California Forest Canopy Cover (LANDFIRE 2024)', fontsize=14, fontweight='bold')
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')

fig.tight_layout()
save_figure(fig, 'landfire_cc/03_statewide_canopy_cover_map')
plt.show()

In [ ]:
# ── Load topography for elevation relationship ────────────────────────────
from src.utils.topo_data_loader import load_topography_downsampled
from src.utils.topo_domain_mappings import classify_elevation, ELEVATION_ZONE_ORDER

elev_ds, elev_transform, elev_meta = load_topography_downsampled()
print(f'Elevation downsampled shape: {elev_ds.shape}')
print(f'CC downsampled shape: {cc_ds.shape}')

# Align arrays to smaller common shape
min_h = min(cc_ds.shape[0], elev_ds.shape[0])
min_w = min(cc_ds.shape[1], elev_ds.shape[1])
cc_aligned = cc_ds[:min_h, :min_w]
elev_aligned = elev_ds[:min_h, :min_w]

# Get valid mask (both have data)
valid_mask = ~np.isnan(cc_aligned) & ~np.isnan(elev_aligned) & (cc_aligned != LANDFIRE_CC_NODATA)
print(f'\nAligned shape: ({min_h}, {min_w}), valid pixels: {valid_mask.sum():,}')

In [ ]:
# ── Figure 04: Canopy Cover by Elevation ──────────────────────────────────
from src.utils.plotting import get_elevation_zone_colors

# Classify elevation into zones
elev_valid = elev_aligned[valid_mask]
cc_valid = cc_aligned[valid_mask]
zones = classify_elevation(elev_valid)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: Box plots of CC by elevation zone
ax = axes[0]
zone_colors = get_elevation_zone_colors()
zone_data = []
zone_labels = []
zone_cols = []
for z in ELEVATION_ZONE_ORDER:
    mask = zones == z
    if mask.sum() > 0:
        zone_data.append(cc_valid[mask])
        zone_labels.append(z)
        zone_cols.append(zone_colors[z])

bp = ax.boxplot(zone_data, labels=zone_labels, patch_artist=True, widths=0.6,
                showfliers=False, medianprops={'color': 'black', 'linewidth': 2})
for patch, color in zip(bp['boxes'], zone_cols):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_xlabel('Elevation Zone')
ax.set_ylabel('Canopy Cover (%)')
ax.set_title('CC Distribution by Elevation Zone')
ax.tick_params(axis='x', rotation=30)

# Right: 2D density scatter
ax2 = axes[1]
# Subsample for scatter (too many points otherwise)
n_sample = min(200_000, len(elev_valid))
idx = np.random.RandomState(42).choice(len(elev_valid), n_sample, replace=False)
ax2.hexbin(elev_valid[idx], cc_valid[idx], gridsize=60, cmap='YlGn', mincnt=1)
ax2.set_xlabel('Elevation (m)')
ax2.set_ylabel('Canopy Cover (%)')
ax2.set_title('CC vs Elevation (hexbin density)')
plt.colorbar(ax2.collections[0], ax=ax2, label='Count')

fig.suptitle('Figure 4: Canopy Cover by Elevation', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
save_figure(fig, 'landfire_cc/04_canopy_cover_by_elevation')
plt.show()

---
## Part 4: Regional Patterns

California's forests are concentrated along the coast (redwoods, mixed evergreen) and in the Sierra Nevada (mixed conifer, fir). The Central Valley and southern deserts are predominantly non-forested.

In [ ]:
# ── Figure 05: Regional Canopy Cover Comparison ───────────────────────────
# Define approximate latitude bands for California regions
# Using pixel rows in the downsampled array (EPSG:3310, origin=upper-left)
# We'll use northing values from the transform

# Calculate northing for each row
northings = np.array([cc_ds_transform[5] + i * cc_ds_transform[4] for i in range(cc_ds.shape[0])])

# Approximate regional boundaries (EPSG:3310 northing in meters)
# These are approximate and based on California geography
regions = {
    'NW Coast / Klamath': (northings > 400_000),
    'Northern Sierra': (northings > 300_000) & (northings <= 400_000),
    'Central Coast / Valley': (northings > 150_000) & (northings <= 300_000),
    'Southern Sierra': (northings > 50_000) & (northings <= 150_000),
    'SoCal / Desert': (northings <= 50_000),
}

fig, ax = plt.subplots(figsize=(14, 7))

region_stats = []
region_names = list(regions.keys())
region_colors_list = ['#2E86C1', '#27AE60', '#F39C12', '#E74C3C', '#8E44AD']

for name, row_mask in regions.items():
    region_slice = cc_ds[row_mask, :]
    valid = region_slice[~np.isnan(region_slice)]
    if len(valid) > 0:
        forested_pct = 100.0 * np.sum(valid > 0) / len(valid)
        mean_cc = np.mean(valid[valid > 0]) if np.sum(valid > 0) > 0 else 0
        region_stats.append({
            'region': name,
            'forested_pct': forested_pct,
            'mean_cc_forested': mean_cc,
            'total_pixels': len(valid)
        })

rdf = pd.DataFrame(region_stats)

x = np.arange(len(rdf))
width = 0.35
bars1 = ax.bar(x - width/2, rdf['forested_pct'], width, label='% Forested',
               color=region_colors_list[:len(rdf)], alpha=0.7, edgecolor='white')
bars2 = ax.bar(x + width/2, rdf['mean_cc_forested'], width, label='Mean CC (forested only)',
               color=region_colors_list[:len(rdf)], alpha=0.4, edgecolor='white', hatch='//')

ax.set_xticks(x)
ax.set_xticklabels(rdf['region'], fontsize=10)
ax.set_ylabel('Percentage')
ax.set_title('Figure 5: Regional Canopy Cover Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

# Add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)

fig.tight_layout()
save_figure(fig, 'landfire_cc/05_regional_canopy_cover_comparison')
plt.show()

In [ ]:
# ── Figure 06: North-South Canopy Cover Gradient ──────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

# Compute mean CC by latitude band (every 10 rows of downsampled)
band_size = 10
n_bands = cc_ds.shape[0] // band_size
lat_means = []
lat_forested = []
lat_positions = []

for i in range(n_bands):
    band = cc_ds[i*band_size:(i+1)*band_size, :]
    valid = band[~np.isnan(band)]
    if len(valid) > 100:  # skip bands with too few valid pixels
        lat_means.append(np.mean(valid))
        lat_forested.append(100.0 * np.sum(valid > 0) / len(valid))
        # Northing of band center
        row_center = i * band_size + band_size // 2
        lat_positions.append(cc_ds_transform[5] + row_center * cc_ds_transform[4])

lat_positions = np.array(lat_positions)
lat_means = np.array(lat_means)
lat_forested = np.array(lat_forested)

# Plot (x = northing, which increases northward)
ax.fill_between(lat_positions, lat_forested, alpha=0.3, color='#27AE60', label='% Forested')
ax.plot(lat_positions, lat_forested, color='#27AE60', linewidth=1.5)

ax2 = ax.twinx()
ax2.plot(lat_positions, lat_means, color='#2E86C1', linewidth=2, label='Mean CC (%)')
ax2.set_ylabel('Mean CC (%)', color='#2E86C1')

ax.set_xlabel('Northing (m, EPSG:3310)')
ax.set_ylabel('% Forested', color='#27AE60')
ax.set_title('Figure 6: North-South Canopy Cover Gradient', fontsize=14, fontweight='bold')

# Add direction labels
ax.annotate('← South', xy=(0.02, 0.02), xycoords='axes fraction', fontsize=10, color='gray')
ax.annotate('North →', xy=(0.92, 0.02), xycoords='axes fraction', fontsize=10, color='gray')

# Combine legends
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

fig.tight_layout()
save_figure(fig, 'landfire_cc/06_north_south_gradient')
plt.show()

In [ ]:
# ── Figure 07: East-West Canopy Profiles ──────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Select 4 latitude bands (northing values)
profile_bands = [
    ('Northern CA (~41°N)', 0.85),
    ('Sacramento (~39°N)', 0.60),
    ('Central CA (~37°N)', 0.40),
    ('Southern CA (~34°N)', 0.15),
]

for ax, (label, frac) in zip(axes.ravel(), profile_bands):
    row = int(frac * cc_ds.shape[0])
    row = min(row, cc_ds.shape[0] - 1)
    
    # Average over a few rows for smoother profile
    row_start = max(0, row - 2)
    row_end = min(cc_ds.shape[0], row + 3)
    profile = np.nanmean(cc_ds[row_start:row_end, :], axis=0)
    
    # X axis: column index (west to east)
    cols = np.arange(len(profile))
    valid = ~np.isnan(profile)
    
    ax.fill_between(cols[valid], profile[valid], alpha=0.4, color='#27AE60')
    ax.plot(cols[valid], profile[valid], color='#0E5C0B', linewidth=0.8)
    ax.set_title(label, fontsize=12)
    ax.set_ylabel('CC (%)')
    ax.set_ylim(-5, 100)
    ax.set_xlabel('← West     Column Index     East →')

fig.suptitle('Figure 7: East-West Canopy Cover Profiles', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
save_figure(fig, 'landfire_cc/07_east_west_canopy_profiles')
plt.show()

---
## Part 5: CC × Fire Perimeters

This section examines canopy cover within historical fire perimeters vs the statewide baseline. Because LANDFIRE CC 2024 reflects current (post-fire) conditions, these comparisons show what canopy cover looks like *after* fires, not what it was *before*.

In [ ]:
# ── Load fire perimeter data ──────────────────────────────────────────────
from src.utils.data_loader import load_fire_perimeters, add_derived_columns

fires_gdf = load_fire_perimeters()
fires_gdf = add_derived_columns(fires_gdf)

# Filter to high-quality era
fires_hq = fires_gdf[fires_gdf['YEAR_'] >= HIGH_QUALITY_CUTOFF].copy()
print(f'CAL FIRE perimeters (1993+): {len(fires_hq):,}')

# Create size categories
fires_hq['size_cat'] = pd.cut(
    fires_hq['GIS_ACRES'],
    bins=[0, 100, 1000, 10000, 100000, float('inf')],
    labels=['<100 ac', '100-1K', '1K-10K', '10K-100K', '>100K']
)

In [ ]:
# ── Create burned mask ────────────────────────────────────────────────────
from rasterio.features import geometry_mask

# Use downsampled CC for burned mask (memory efficient)
h_ds, w_ds = cc_ds.shape

# Rasterize all HQ fire perimeters
fires_3310 = fires_hq.to_crs(CRS_ALBERS)
burned_mask = geometry_mask(
    fires_3310.geometry.values,
    out_shape=(h_ds, w_ds),
    transform=cc_ds_transform,
    invert=True  # True = burned
)

valid_ds = ~np.isnan(cc_ds)
burned_pixels = cc_ds[burned_mask & valid_ds]
unburned_pixels = cc_ds[~burned_mask & valid_ds]

print(f'Burned pixels (downsampled): {len(burned_pixels):,}')
print(f'Unburned pixels (downsampled): {len(unburned_pixels):,}')
print(f'Burned mean CC: {np.mean(burned_pixels):.1f}%')
print(f'Unburned mean CC: {np.mean(unburned_pixels):.1f}%')

In [ ]:
# ── Figure 08: Burned vs Unburned CC ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: KDE comparison
ax = axes[0]
# Subsample for KDE
rng = np.random.RandomState(42)
n_kde = 100_000
burned_sample = rng.choice(burned_pixels, min(n_kde, len(burned_pixels)), replace=False)
unburned_sample = rng.choice(unburned_pixels, min(n_kde, len(unburned_pixels)), replace=False)

sns.kdeplot(unburned_sample, ax=ax, label='Unburned', color='#2E86C1', linewidth=2, fill=True, alpha=0.3)
sns.kdeplot(burned_sample, ax=ax, label='Burned (1993+)', color='#E74C3C', linewidth=2, fill=True, alpha=0.3)
ax.set_xlabel('Canopy Cover (%)')
ax.set_ylabel('Density')
ax.set_title('CC Distribution: Burned vs Unburned')
ax.legend(fontsize=11)

# Right: Stacked bar by broad category
ax2 = axes[1]
broad_colors_map = get_canopy_broad_colors()
burned_broad = classify_canopy_cover(burned_pixels)
unburned_broad = classify_canopy_cover(unburned_pixels)

categories = CC_BROAD_CATEGORY_ORDER
burned_pcts = [100.0 * np.sum(burned_broad == c) / len(burned_broad) for c in categories]
unburned_pcts = [100.0 * np.sum(unburned_broad == c) / len(unburned_broad) for c in categories]

x = np.arange(len(categories))
width = 0.35
ax2.bar(x - width/2, unburned_pcts, width, label='Unburned',
        color=[broad_colors_map[c] for c in categories], alpha=0.6, edgecolor='white')
ax2.bar(x + width/2, burned_pcts, width, label='Burned',
        color=[broad_colors_map[c] for c in categories], alpha=1.0, edgecolor='#E74C3C', linewidth=1.5)
ax2.set_xticks(x)
ax2.set_xticklabels(categories, rotation=30, ha='right', fontsize=9)
ax2.set_ylabel('% of Pixels')
ax2.set_title('Broad CC Categories: Burned vs Unburned')
ax2.legend(fontsize=11)

fig.suptitle('Figure 8: Burned vs Unburned Canopy Cover', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
save_figure(fig, 'landfire_cc/08_burned_vs_unburned_cc')
plt.show()

In [ ]:
# ── Extract CC for fire perimeters (subsample for speed) ──────────────────
# Use downsampled CC for polygon extraction
# Sample up to 500 fires for polygon stats (full extraction is slow)
n_sample_fires = min(500, len(fires_hq))
fires_sample = fires_hq.sample(n_sample_fires, random_state=42).copy()

print(f'Extracting CC for {n_sample_fires} fire perimeters...')
cc_stats = extract_cc_for_polygons(
    cc_ds, cc_ds_transform, fires_sample.to_crs(CRS_ALBERS), crs=CRS_ALBERS
)

fires_sample['mean_cc'] = cc_stats['mean_cc_pct']
fires_sample['forested_frac'] = cc_stats['forested_fraction']
fires_sample['dominant_cc'] = cc_stats['dominant_cc']

valid_stats = fires_sample.dropna(subset=['mean_cc'])
print(f'Valid CC extractions: {len(valid_stats)}')

In [ ]:
# ── Figure 09: CC by Fire Size Category ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Mean CC by size category
ax = axes[0]
size_order = ['<100 ac', '100-1K', '1K-10K', '10K-100K', '>100K']
size_colors = ['#2ECC71', '#82E0AA', '#F9E79F', '#E74C3C', '#922B21']
size_stats = valid_stats.groupby('size_cat', observed=True)['mean_cc'].agg(['mean', 'median', 'count'])
size_stats = size_stats.reindex(size_order)

bars = ax.bar(range(len(size_stats)), size_stats['mean'], color=size_colors[:len(size_stats)],
              edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(size_stats)))
ax.set_xticklabels(size_stats.index, fontsize=10)
ax.set_ylabel('Mean CC (%)')
ax.set_title('Mean Canopy Cover by Fire Size')
for bar, (_, row) in zip(bars, size_stats.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{row["mean"]:.1f}%\n(n={int(row["count"])})',
            ha='center', va='bottom', fontsize=8)

# Right: Forested fraction by size category
ax2 = axes[1]
ff_stats = valid_stats.groupby('size_cat', observed=True)['forested_frac'].agg(['mean', 'median'])
ff_stats = ff_stats.reindex(size_order)
bars2 = ax2.bar(range(len(ff_stats)), 100*ff_stats['mean'], color=size_colors[:len(ff_stats)],
                edgecolor='white', linewidth=0.5)
ax2.set_xticks(range(len(ff_stats)))
ax2.set_xticklabels(ff_stats.index, fontsize=10)
ax2.set_ylabel('Mean Forested Fraction (%)')
ax2.set_title('Forested Fraction by Fire Size')

fig.suptitle('Figure 9: Canopy Cover by Fire Size Category', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
save_figure(fig, 'landfire_cc/09_cc_by_fire_size_category')
plt.show()

In [ ]:
# ── Figure 10: CC in Fire Perimeters Over Time ────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

# Group by year
yearly_cc = valid_stats.groupby('YEAR_')['mean_cc'].agg(['mean', 'median', 'count'])
yearly_cc = yearly_cc[yearly_cc['count'] >= 3]  # Minimum 3 fires per year

ax.bar(yearly_cc.index, yearly_cc['mean'], color='#27AE60', alpha=0.6, label='Mean CC in fire perimeters')
ax.plot(yearly_cc.index, yearly_cc['median'], 'o-', color='#0E5C0B', linewidth=2,
        markersize=5, label='Median CC')

# Add trend line
from numpy.polynomial import polynomial as P
x_fit = yearly_cc.index.values.astype(float)
coefs = P.polyfit(x_fit, yearly_cc['mean'].values, 1)
trend = P.polyval(x_fit, coefs)
ax.plot(x_fit, trend, '--', color='#E74C3C', linewidth=2, label='Linear trend')

ax.set_xlabel('Year')
ax.set_ylabel('Mean CC in Fire Perimeters (%)')
ax.set_title('Figure 10: Canopy Cover in Fire Perimeters Over Time', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.annotate('Note: CC is 2024 snapshot — reflects post-fire regrowth/loss',
            xy=(0.02, 0.02), xycoords='axes fraction', fontsize=9, fontstyle='italic', color='gray')

fig.tight_layout()
save_figure(fig, 'landfire_cc/10_cc_fire_perimeters_temporal')
plt.show()

---
## Part 6: CC × Burn Severity (MTBS)

This section examines the relationship between current canopy cover and historical burn severity from MTBS. Higher severity fires should result in lower current canopy cover (for recent fires).

In [ ]:
# ── Load MTBS data ────────────────────────────────────────────────────────
from src.utils.mtbs_data_loader import load_mtbs_fod, add_mtbs_derived_columns, reproject_to_albers as mtbs_reproject

mtbs_gdf = load_mtbs_fod(state='CA')
mtbs_gdf = add_mtbs_derived_columns(mtbs_gdf)
mtbs_gdf = mtbs_reproject(mtbs_gdf)

# Filter to fires with severity data
mtbs_sev = mtbs_gdf[mtbs_gdf['has_severity']].copy()
print(f'MTBS CA fires with severity: {len(mtbs_sev)}')

# Extract CC at MTBS centroids
mtbs_sev['cc_value'] = extract_cc_at_points(
    cc_ds, cc_ds_transform, mtbs_sev, crs=CRS_ALBERS
)
mtbs_sev_valid = mtbs_sev.dropna(subset=['cc_value']).copy()
print(f'MTBS fires with valid CC: {len(mtbs_sev_valid)}')

In [ ]:
# ── Figure 11: Severity vs Canopy Cover ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: Scatter of High_T severity threshold vs CC
ax = axes[0]
sc = ax.scatter(
    mtbs_sev_valid['cc_value'],
    mtbs_sev_valid['High_T_clean'],
    c=mtbs_sev_valid['Ig_Year'],
    cmap='viridis', alpha=0.6, s=30, edgecolors='white', linewidth=0.3
)
plt.colorbar(sc, ax=ax, label='Ignition Year')
ax.set_xlabel('Current CC at Fire Centroid (%)')
ax.set_ylabel('MTBS High Severity Threshold (dNBR)')
ax.set_title('Burn Severity vs Current CC')

# Right: Box plot of CC by severity category
ax2 = axes[1]
# Create severity bins from High_T
mtbs_sev_valid['sev_cat'] = pd.cut(
    mtbs_sev_valid['High_T_clean'],
    bins=[0, 200, 400, 600, 1000],
    labels=['Low\n(<200)', 'Moderate\n(200-400)', 'High\n(400-600)', 'Very High\n(>600)']
)
sev_colors = ['#F1C40F', '#E67E22', '#E74C3C', '#922B21']
sev_data = []
sev_labels = []
for cat in ['Low\n(<200)', 'Moderate\n(200-400)', 'High\n(400-600)', 'Very High\n(>600)']:
    vals = mtbs_sev_valid.loc[mtbs_sev_valid['sev_cat'] == cat, 'cc_value'].dropna()
    if len(vals) > 0:
        sev_data.append(vals.values)
        sev_labels.append(cat)

bp = ax2.boxplot(sev_data, labels=sev_labels, patch_artist=True, widths=0.6,
                 showfliers=False, medianprops={'color': 'black', 'linewidth': 2})
for patch, color in zip(bp['boxes'], sev_colors[:len(sev_data)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax2.set_ylabel('Current CC at Centroid (%)')
ax2.set_title('Current CC by Severity Category')

fig.suptitle('Figure 11: Burn Severity vs Canopy Cover', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
save_figure(fig, 'landfire_cc/11_severity_vs_canopy_cover')
plt.show()

In [ ]:
# ── Figure 12: CC by Fire Season ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: CC distribution by fire season (MTBS)
ax = axes[0]
season_order = ['High Risk', 'Transition', 'Low Risk']
season_colors = ['#E74C3C', '#F39C12', '#2E86C1']
season_data = []
season_labels_plot = []
for s, c in zip(season_order, season_colors):
    vals = mtbs_sev_valid.loc[mtbs_sev_valid['Fire_Season'] == s, 'cc_value'].dropna()
    if len(vals) > 0:
        season_data.append(vals.values)
        season_labels_plot.append(f'{s}\n(n={len(vals)})')

vp = ax.violinplot(season_data, showmedians=True, showextrema=False)
for i, body in enumerate(vp['bodies']):
    body.set_facecolor(season_colors[i])
    body.set_alpha(0.6)
ax.set_xticks(range(1, len(season_labels_plot) + 1))
ax.set_xticklabels(season_labels_plot)
ax.set_ylabel('CC at Fire Centroid (%)')
ax.set_title('CC by Fire Season (MTBS Fires)')

# Right: Monthly mean CC for fires
ax2 = axes[1]
monthly_cc = mtbs_sev_valid.groupby('Ig_Month')['cc_value'].agg(['mean', 'count'])
monthly_cc = monthly_cc[monthly_cc['count'] >= 5]
months = monthly_cc.index
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# Color by season
month_colors = []
for m in months:
    if m in [6, 7, 8, 9]:
        month_colors.append('#E74C3C')
    elif m in [4, 5, 10, 11]:
        month_colors.append('#F39C12')
    else:
        month_colors.append('#2E86C1')

ax2.bar(months, monthly_cc['mean'], color=month_colors, edgecolor='white')
ax2.set_xticks(range(1, 13))
ax2.set_xticklabels(month_labels, fontsize=9)
ax2.set_ylabel('Mean CC at Fire Centroid (%)')
ax2.set_title('Mean CC by Ignition Month')

fig.suptitle('Figure 12: Canopy Cover by Fire Season', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
save_figure(fig, 'landfire_cc/12_cc_by_fire_season')
plt.show()

In [ ]:
# ── Figure 13: CC-Severity Trend by Decade ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Mean CC by decade
ax = axes[0]
decade_cc = mtbs_sev_valid.groupby('Ig_Decade')['cc_value'].agg(['mean', 'median', 'count'])
decade_cc = decade_cc[decade_cc['count'] >= 10]
decade_colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(decade_cc)))

bars = ax.bar(range(len(decade_cc)), decade_cc['mean'], color=decade_colors, edgecolor='white')
ax.set_xticks(range(len(decade_cc)))
ax.set_xticklabels([f"{int(d)}s" for d in decade_cc.index], fontsize=10)
ax.set_ylabel('Mean CC at Fire Centroid (%)')
ax.set_title('Mean CC at Fire Centroids by Decade')
for bar, (_, row) in zip(bars, decade_cc.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'n={int(row["count"])}', ha='center', va='bottom', fontsize=8)

# Right: CC vs severity by decade
ax2 = axes[1]
decades = sorted(mtbs_sev_valid['Ig_Decade'].dropna().unique())
for i, dec in enumerate(decades):
    mask = mtbs_sev_valid['Ig_Decade'] == dec
    sub = mtbs_sev_valid[mask]
    if len(sub) >= 10:
        ax2.scatter(sub['cc_value'], sub['High_T_clean'],
                    alpha=0.5, s=20, label=f"{int(dec)}s (n={len(sub)})",
                    color=plt.cm.viridis(i / max(1, len(decades)-1)))

ax2.set_xlabel('Current CC (%)')
ax2.set_ylabel('High Severity Threshold (dNBR)')
ax2.set_title('CC vs Severity by Decade')
ax2.legend(fontsize=8, loc='upper right')

fig.suptitle('Figure 13: CC-Severity Trends by Decade', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
save_figure(fig, 'landfire_cc/13_cc_severity_by_decade')
plt.show()

---
## Part 7: CC × Fire Size & Frequency

Does canopy cover correlate with fire frequency or size? Denser forests may support larger, more intense fires, while open areas may see more frequent but smaller fires.

In [ ]:
# ── Figure 14: Fire Frequency by CC Class ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Compute fire density per CC class using the burned mask
ax = axes[0]
cc_class_fire_counts = []
cc_class_total_counts = []
cc_labels_plot = []

for v in CC_ORDERED_VALUES:
    # Pixels with this CC value
    if v == 0:
        class_mask = (cc_ds == 0) & valid_ds
    else:
        # For midpoint values, match the exact value
        class_mask = (cc_ds == v) & valid_ds
    
    total = class_mask.sum()
    burned = (class_mask & burned_mask).sum()
    cc_class_fire_counts.append(burned)
    cc_class_total_counts.append(total)
    cc_labels_plot.append(get_cc_label(v))

# Burn rate = burned / total
cc_class_total = np.array(cc_class_total_counts, dtype=float)
cc_class_fire = np.array(cc_class_fire_counts, dtype=float)
burn_rate = np.where(cc_class_total > 0, 100.0 * cc_class_fire / cc_class_total, 0)

cc_colors_map = get_canopy_cover_colors()
bar_colors = [cc_colors_map.get(v, '#888') for v in CC_ORDERED_VALUES]
bars = ax.bar(range(len(CC_ORDERED_VALUES)), burn_rate, color=bar_colors, edgecolor='white')
ax.set_xticks(range(len(CC_ORDERED_VALUES)))
ax.set_xticklabels(cc_labels_plot, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Burn Rate (% of pixels burned 1993+)')
ax.set_title('Historical Burn Rate by CC Class')

# Right: Number of fire perimeters by CC
ax2 = axes[1]
# Classify dominant CC of fire perimeters
dom_cc = valid_stats['dominant_cc'].dropna()
dom_counts = dom_cc.value_counts().sort_index()

dom_x = []
dom_y = []
dom_c = []
for v in CC_ORDERED_VALUES:
    dom_x.append(v)
    dom_y.append(dom_counts.get(v, 0))
    dom_c.append(cc_colors_map.get(v, '#888'))

ax2.bar(range(len(dom_x)), dom_y, color=dom_c, edgecolor='white')
ax2.set_xticks(range(len(dom_x)))
ax2.set_xticklabels([get_cc_label(v) for v in dom_x], rotation=45, ha='right', fontsize=9)
ax2.set_ylabel('Number of Fire Perimeters')
ax2.set_title('Dominant CC Class in Fire Perimeters')

fig.suptitle('Figure 14: Fire Frequency by CC Class', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
save_figure(fig, 'landfire_cc/14_fire_frequency_by_cc_class')
plt.show()

In [ ]:
# ── Figure 15: Fire Size Distribution by CC Category ──────────────────────
fig, ax = plt.subplots(figsize=(14, 7))

# Classify fire perimeters by their CC broad category
valid_stats['cc_broad'] = classify_canopy_cover(valid_stats['mean_cc'].values)

broad_data = []
broad_labels_plot = []
broad_c = []
broad_colors_map = get_canopy_broad_colors()

for cat in CC_BROAD_CATEGORY_ORDER:
    vals = valid_stats.loc[valid_stats['cc_broad'] == cat, 'GIS_ACRES'].dropna()
    if len(vals) >= 5:
        broad_data.append(np.log10(vals.values + 1))
        broad_labels_plot.append(f'{cat}\n(n={len(vals)})')
        broad_c.append(broad_colors_map[cat])

vp = ax.violinplot(broad_data, showmedians=True, showextrema=False)
for i, body in enumerate(vp['bodies']):
    body.set_facecolor(broad_c[i])
    body.set_alpha(0.7)
    body.set_edgecolor('white')

ax.set_xticks(range(1, len(broad_labels_plot) + 1))
ax.set_xticklabels(broad_labels_plot, fontsize=10)
ax.set_ylabel('log₁₀(Fire Size in Acres)')
ax.set_title('Figure 15: Fire Size Distribution by Canopy Cover Category', fontsize=14, fontweight='bold')

# Add reference lines
for val, label in [(2, '100 ac'), (3, '1K ac'), (4, '10K ac'), (5, '100K ac')]:
    ax.axhline(val, color='gray', linestyle=':', alpha=0.4)
    ax.text(len(broad_data) + 0.6, val, label, fontsize=8, color='gray', va='center')

fig.tight_layout()
save_figure(fig, 'landfire_cc/15_fire_size_by_canopy_cover')
plt.show()

---
## Part 8: Spatial Visualization

Detailed spatial views of canopy cover with fire overlay for regional context.

In [ ]:
# ── Figure 16: Regional Detail Map ────────────────────────────────────────
# Zoomed view of Northern California showing CC + fire outlines
fig, axes = plt.subplots(1, 2, figsize=(18, 10))

cmap = build_cc_colormap()
norm = build_cc_norm()

# Define two zoom regions
zoom_regions = [
    {'title': 'NW California (Klamath-Mendocino)',
     'row_frac': (0.0, 0.35), 'col_frac': (0.0, 0.5)},
    {'title': 'Central Sierra Nevada',
     'row_frac': (0.35, 0.65), 'col_frac': (0.45, 1.0)},
]

for ax, region in zip(axes, zoom_regions):
    r0 = int(region['row_frac'][0] * cc_ds.shape[0])
    r1 = int(region['row_frac'][1] * cc_ds.shape[0])
    c0 = int(region['col_frac'][0] * cc_ds.shape[1])
    c1 = int(region['col_frac'][1] * cc_ds.shape[1])
    
    sub = cc_ds[r0:r1, c0:c1]
    
    # Compute extent for this subset
    sub_left = cc_ds_transform[2] + c0 * abs(cc_ds_transform[0])
    sub_top = cc_ds_transform[5] + r0 * cc_ds_transform[4]
    sub_right = cc_ds_transform[2] + c1 * abs(cc_ds_transform[0])
    sub_bottom = cc_ds_transform[5] + r1 * cc_ds_transform[4]
    sub_extent = [sub_left, sub_right, sub_bottom, sub_top]
    
    ax.imshow(sub, cmap=cmap, norm=norm, extent=sub_extent, origin='upper', interpolation='nearest')
    
    # Overlay fire perimeters that intersect this region
    from shapely.geometry import box
    region_box = box(sub_left, sub_bottom, sub_right, sub_top)
    fires_in_region = fires_3310[fires_3310.geometry.intersects(region_box)]
    if len(fires_in_region) > 0:
        fires_in_region.boundary.plot(ax=ax, color='red', linewidth=0.5, alpha=0.6)
    
    try:
        ctx.add_basemap(ax, crs=CRS_ALBERS, source=ctx.providers.CartoDB.PositronNoLabels, alpha=0.2)
    except Exception:
        pass
    
    ax.set_title(region['title'], fontsize=12)
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')

fig.suptitle('Figure 16: Regional CC Detail with Fire Perimeters', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
save_figure(fig, 'landfire_cc/16_regional_detail_map')
plt.show()

In [ ]:
# ── Figure 17: Statewide CC + Fire Overlay ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 14))

# Left: CC map with fire perimeters
ax = axes[0]
h_ds, w_ds = cc_ds.shape
left = cc_ds_transform[2]
top = cc_ds_transform[5]
right = left + w_ds * abs(cc_ds_transform[0])
bottom = top + h_ds * cc_ds_transform[4]
extent = [left, right, bottom, top]

ax.imshow(cc_ds, cmap=cmap, norm=norm, extent=extent, origin='upper', interpolation='nearest')

# Overlay large fire perimeters (>10K acres)
large_fires = fires_3310[fires_3310['GIS_ACRES'] >= 10000]
if len(large_fires) > 0:
    large_fires.boundary.plot(ax=ax, color='red', linewidth=0.3, alpha=0.5)

try:
    ctx.add_basemap(ax, crs=CRS_ALBERS, source=ctx.providers.CartoDB.PositronNoLabels, alpha=0.2)
except Exception:
    pass

ax.set_title('Canopy Cover + Fire Perimeters (≥10K acres)', fontsize=12)
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')

# Right: Burned area mask colored by CC
ax2 = axes[1]
# Show only burned pixels
cc_burned_only = cc_ds.copy()
cc_burned_only[~burned_mask | ~valid_ds] = np.nan

ax2.imshow(cc_burned_only, cmap=cmap, norm=norm, extent=extent, origin='upper', interpolation='nearest')

try:
    ctx.add_basemap(ax2, crs=CRS_ALBERS, source=ctx.providers.CartoDB.PositronNoLabels, alpha=0.3)
except Exception:
    pass

ax2.set_title('CC Within Burned Areas Only (1993+)', fontsize=12)
ax2.set_xlabel('Easting (m)')
ax2.set_ylabel('Northing (m)')

# Add shared colorbar
from matplotlib.cm import ScalarMappable
sm = ScalarMappable(cmap=cmap, norm=norm)
cbar = fig.colorbar(sm, ax=axes, shrink=0.4, pad=0.02, orientation='horizontal')
cbar.set_ticks(CC_ORDERED_VALUES)
cbar.set_ticklabels([get_cc_label(v) for v in CC_ORDERED_VALUES])

fig.suptitle('Figure 17: Statewide CC + Fire Overlay', fontsize=14, fontweight='bold', y=0.98)
fig.tight_layout(rect=[0, 0.05, 1, 0.97])
save_figure(fig, 'landfire_cc/17_cc_fire_overlay_map')
plt.show()

In [ ]:
# ── Interactive Folium Map ────────────────────────────────────────────────
import folium
from folium.plugins import MarkerCluster

print('Creating interactive Folium map...')

# Center on California
m = folium.Map(location=[37.5, -119.5], zoom_start=6, tiles='CartoDB positron')

# Add MTBS fire markers colored by CC value
mtbs_wgs84 = mtbs_sev_valid.to_crs(CRS_WGS84)
marker_cluster = MarkerCluster(name='MTBS Fires (colored by CC)').add_to(m)

# Color markers by CC category
cc_marker_colors = {
    'Non-Forested': 'gray',
    'Open (10-30%)': 'lightgreen',
    'Moderate (30-60%)': 'green',
    'Dense (60-80%)': 'darkgreen',
    'Very Dense (80-100%)': 'black',
    'Unknown': 'blue',
}

for _, row in mtbs_wgs84.iterrows():
    cc_val = row['cc_value']
    cc_cat = classify_canopy_cover(int(cc_val)) if not np.isnan(cc_val) else 'Unknown'
    color = cc_marker_colors.get(cc_cat, 'blue')
    popup_text = (
        f"<b>{row.get('Incid_Name', 'Unknown')}</b><br>"
        f"Year: {row.get('Ig_Year', 'N/A')}<br>"
        f"Acres: {row.get('BurnBndAc', 0):,.0f}<br>"
        f"CC: {cc_val:.0f}% ({cc_cat})<br>"
        f"High_T: {row.get('High_T_clean', 'N/A')}"
    )
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5, color=color, fill=True, fill_color=color,
        fill_opacity=0.7, popup=folium.Popup(popup_text, max_width=250)
    ).add_to(marker_cluster)

# Add legend
legend_html = '''
<div style="position:fixed; bottom:50px; left:50px; z-index:1000;
     background:white; padding:10px; border:2px solid grey; border-radius:5px;">
<b>CC at Fire Centroid</b><br>
<i style="background:gray; width:12px; height:12px; display:inline-block;"></i> Non-Forested<br>
<i style="background:lightgreen; width:12px; height:12px; display:inline-block;"></i> Open (10-30%)<br>
<i style="background:green; width:12px; height:12px; display:inline-block;"></i> Moderate (30-60%)<br>
<i style="background:darkgreen; width:12px; height:12px; display:inline-block;"></i> Dense (60-80%)<br>
<i style="background:black; width:12px; height:12px; display:inline-block;"></i> Very Dense (80-100%)<br>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl().add_to(m)

# Save
interactive_dir = OUTPUT_DIR / 'interactive'
interactive_dir.mkdir(parents=True, exist_ok=True)
map_path = interactive_dir / 'landfire_cc_interactive.html'
m.save(str(map_path))
print(f'Interactive map saved: {map_path}')
m

---
## Part 9: ML Features & Summary

This section demonstrates how CC data will be aggregated to the 800m ML grid and summarizes the key findings.

In [ ]:
# ── 800m Grid Aggregation Demo ────────────────────────────────────────────
# Demonstrate CC aggregation at the target ML resolution

from rasterio.warp import reproject as warp_reproject
from rasterio.warp import calculate_default_transform as calc_transform

# Target: 800m grid
target_res = 800  # meters
h_full, w_full = cc_array.shape

# Calculate output dimensions
pixel_size = abs(cc_transform[0])
agg_factor = target_res / pixel_size
out_h = int(h_full / agg_factor)
out_w = int(w_full / agg_factor)

print(f'Full resolution: {h_full} x {w_full} at {pixel_size:.1f}m')
print(f'800m grid: {out_h} x {out_w} ({out_h * out_w:,} cells)')

# Convert to midpoint floats for mean aggregation
cc_float = cc_array.astype(np.float32)
cc_float[cc_float == LANDFIRE_CC_NODATA] = np.nan

# Downsample to ~800m using block averaging
# Quick approach: reshape and mean
block_h = int(agg_factor)
block_w = int(agg_factor)
trim_h = out_h * block_h
trim_w = out_w * block_w

cc_trimmed = cc_float[:trim_h, :trim_w]
cc_800m = np.nanmean(
    cc_trimmed.reshape(out_h, block_h, out_w, block_w),
    axis=(1, 3)
)

print(f'\n800m aggregated CC:')
print(f'  Shape: {cc_800m.shape}')
print(f'  Valid cells: {np.sum(np.isfinite(cc_800m)):,}')
print(f'  Mean CC: {np.nanmean(cc_800m):.1f}%')
print(f'  Median CC: {np.nanmedian(cc_800m):.1f}%')

# Also compute forested fraction at 800m
# Need binary forested/not mask
cc_binary = (cc_array > 0).astype(np.float32)
cc_binary[cc_array == LANDFIRE_CC_NODATA] = np.nan
cc_binary_trimmed = cc_binary[:trim_h, :trim_w]
forested_800m = np.nanmean(
    cc_binary_trimmed.reshape(out_h, block_h, out_w, block_w),
    axis=(1, 3)
)

print(f'  Mean forested fraction: {np.nanmean(forested_800m):.3f}')

# Free large arrays
del cc_float, cc_binary, cc_trimmed, cc_binary_trimmed

In [ ]:
# ── ML Feature Summary ────────────────────────────────────────────────────
print('\n' + '=' * 70)
print('LANDFIRE CC — ML Feature Engineering Summary')
print('=' * 70)
print()

features = [
    ('mean_cc_pct', 'Mean canopy cover percentage in 800m cell', 'Continuous [0-95]'),
    ('forested_fraction', 'Fraction of 30m pixels with CC > 0', 'Continuous [0-1]'),
    ('dominant_cc_class', 'Most common CC class in 800m cell', 'Categorical [0,15,...,95]'),
    ('cc_broad_category', 'Broad CC category (5 groups)', 'Categorical'),
    ('cc_heterogeneity', 'Std deviation of CC within 800m cell', 'Continuous'),
    ('cc_edge_density', 'Proportion of cells with mixed forest/non-forest', 'Continuous [0-1]'),
]

print(f'{"Feature":<25} {"Description":<50} {"Type"}')
print('-' * 95)
for name, desc, ftype in features:
    print(f'{name:<25} {desc:<50} {ftype}')

print()
print('Temporal caveat: LANDFIRE CC 2024 is a static snapshot. For')
print('historical fire prediction, CC represents post-disturbance')
print('conditions for fires 2015-2024 and pre-disturbance conditions')
print('for earlier fires (pre-2015 canopy was different).')

---
## Executive Summary

### Key Findings

1. **California is predominantly non-forested**: The majority of the state (Central Valley, Mojave Desert, urban areas) has 0% tree canopy cover
2. **Dense forests concentrated in specific regions**: NW coast (Klamath-Mendocino), Sierra Nevada, and scattered coastal ranges
3. **Strong elevation-CC relationship**: Canopy cover peaks in the montane zones (900-2700m) and drops off in alpine and valley areas
4. **Fire-CC interaction**: Burned areas show different CC distributions than unburned areas, reflecting both fire preferences for certain cover types and post-fire canopy loss
5. **Recent high-severity fires show lower current CC**: Expected pattern — fires that burned more severely left less canopy
6. **CC is a valuable ML feature**: Provides fuel density, crown fire potential, and landscape structure information

### Temporal Caveat

LANDFIRE CC 2024 incorporates disturbances through 2024. This means:
- For fires 2015-2024: CC reflects **post-fire** conditions
- For fires pre-2015: CC reflects **current** conditions (some recovery)
- For ML features: Use CC as a **static landscape descriptor**, not as pre-fire conditions

### Next Steps

1. **Phase 2**: Aggregate CC to 800m grid alongside topography
2. **Phase 3**: Integrate with PRISM climate data and fuel moisture
3. **Consider**: LANDFIRE CC change detection products for temporal CC variation
4. **Cross-reference**: LANDFIRE Existing Vegetation Type (EVT) for species-level fuel characterization